In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.preprocessing import LabelEncoder, StandardScaler
import matplotlib.pyplot as plt
import lab_utils

cardio = pd.read_csv('../01-feature-importance-and-selection/data/cardiovascular_risk.csv')
credit = pd.read_csv('../01-feature-importance-and-selection/data/credit_risk.csv')
credit['loan_purpose'] = LabelEncoder().fit_transform(credit['loan_purpose'])

artifacts = lab_utils.load_upstream_artifacts()
features = artifacts['feature_sets']['cardiovascular_risk']['set_A']
X = cardio[features]
y = cardio['target']

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000).fit(X_s, y)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_s, y)

print("Готово")

In [ ]:
# Ячейка 2: Важности
results = []

# LR coefficients
for f, c in zip(features, lr.coef_[0]):
    results.append({'dataset':'cardio','model':'LR','feature_set':'set_A','method':'native_coef',
                    'feature':f,'score':abs(c),'rank':0})

# RF importance
for f, imp in zip(features, rf.feature_importances_):
    results.append({'dataset':'cardio','model':'RF','feature_set':'set_A','method':'native_importance',
                    'feature':f,'score':imp,'rank':0})

# Permutation
for model, name in [(lr,'LR'),(rf,'RF')]:
    perm = permutation_importance(model, X_s, y, n_repeats=5, random_state=42)
    for f, imp in zip(features, perm.importances_mean):
        results.append({'dataset':'cardio','model':name,'feature_set':'set_A',f'method':'permutation_{name}',
                        'feature':f,'score':imp,'rank':0})

gi = pd.DataFrame(results)
gi['rank'] = gi.groupby('method')['score'].rank(ascending=False)
gi.to_csv('outputs/global_importance_comparison.csv', index=False)
print(gi.pivot_table(values='score', index='feature', columns='method'))

In [ ]:
# Ячейка 3: Partial dependence
pd_data = []
for f in features:
    pd_res = partial_dependence(rf, X_s, features=[f], kind='average')
    grid = pd_res['grid_values'][0]
    scores = pd_res['average'][0]
    pd_data.append({
        'dataset':'cardio','model':'RF','feature_set':'set_A','raw_feature':f,
        'grid_min':grid.min(),'grid_max':grid.max(),
        'score_min':scores.min(),'score_max':scores.max(),
        'score_delta':scores.max()-scores.min(),
        'trend':'up' if scores[-1]>scores[0] else 'down'
    })
    plt.plot(grid, scores)
    plt.title(f)
    plt.show()

pd.DataFrame(pd_data).to_csv('outputs/partial_dependence_summary.csv', index=False)